<a href="https://colab.research.google.com/github/krisshattanicole/Acode/blob/main/Flask_Orchestrator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import subprocess
import time
from flask import Flask, request, jsonify

!pip install flask-cors
from flask_cors import CORS

app = Flask(__name__)
# Enable CORS for Chrome Extension origin
CORS(app, resources={r"/api/*": {"origins": "*"}})

# --- CORE CONFIGURATION ---
VERSION = "3.0.0-PRO"
API_KEY = os.getenv("KN3AUX_API_KEY", "")

# --- HELPER FUNCTIONS ---

def verify_key():
    """Validates the X-API-Key header if a key is configured."""
    if not API_KEY:
        return True
    return request.headers.get('X-API-Key') == API_KEY

def run_cmd(cmd, timeout=15):
    """Executes system commands safely and returns output."""
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return result.stdout.strip(), result.stderr.strip()
    except subprocess.TimeoutExpired:
        return "", "Command timed out"
    except Exception as e:
        return "", str(e)

# --- MIDDLEWARE ---

@app.before_request
def gatekeeper():
    if not verify_key():
        return jsonify({"error": "Unauthorized: Invalid API Key"}), 401

# --- HEALTH & SYSTEM ---

@app.route('/api/health', methods=['GET'])
def health_check():
    return jsonify({
        "status": "online",
        "version": VERSION,
        "platform": "Termux/Android",
        "timestamp": time.time()
    })

# --- DROIDHUB / USB ENDPOINTS ---

@app.route('/api/tools/usb', methods=['GET'])
def list_usb():
    stdout, stderr = run_cmd("lsusb")
    if stderr and not stdout:
        return jsonify({"error": stderr}), 500

    devices = []
    for line in stdout.split('\n'):
        if not line: continue
        parts = line.split()
        if len(parts) >= 6:
            devices.append({
                "bus": parts[1],
                "device": parts[3].rstrip(':'),
                "id": parts[5],
                "info": " ".join(parts[6:])
            })
    return jsonify(devices)

@app.route('/api/tools/usb/info', methods=['POST'])
def usb_info():
    data = request.json
    device_id = data.get('device') # e.g., "18d1:4ee7"
    if not device_id:
        return jsonify({"error": "No device ID provided"}), 400

    vendor, product = device_id.split(':')
    stdout, _ = run_cmd(f"lsusb -d {vendor}:{product} -v")
    return jsonify({"raw_details": stdout})

@app.route('/api/tools/usb/watcher', methods=['POST'])
def start_watcher():
    interval = request.json.get('interval', 3)
    # Background logic would be triggered here
    return jsonify({"status": "watcher_started", "interval": interval})

@app.route('/api/tools/usb/watcher', methods=['DELETE'])
def stop_watcher():
    return jsonify({"status": "watcher_stopped"})

# --- RESCUE / ADB / FASTBOOT ---

@app.route('/api/tools/rescue', methods=['GET'])
def rescue_scan():
    """Unified scan for ADB and Fastboot devices."""
    adb_out, _ = run_cmd("adb devices")
    fb_out, _ = run_cmd("fastboot devices")

    devices = []
    # Parse ADB
    for line in adb_out.split('\n')[1:]:
        if '\tdevice' in line:
            devices.append({"serial": line.split('\t')[0], "type": "ADB", "status": "online"})
    # Parse Fastboot
    for line in fb_out.split('\n'):
        if line:
            devices.append({"serial": line.split('\t')[0], "type": "FASTBOOT", "status": "bootloader"})

    return jsonify({"devices": devices, "count": len(devices)})

@app.route('/api/tools/rescue/blind-tap', methods=['POST'])
def blind_tap():
    data = request.json
    x, y = data.get('x', 500), data.get('y', 500)
    stdout, stderr = run_cmd(f"adb shell input tap {x} {y}")
    if stderr:
        return jsonify({"error": stderr}), 400
    return jsonify({"status": "tapped", "coords": [x, y]})

@app.route('/api/tools/rescue/fastboot', methods=['POST'])
def fastboot_action():
    action = request.json.get('action') # e.g., "getvar all", "reboot"
    stdout, stderr = run_cmd(f"fastboot {action}")
    return jsonify({"stdout": stdout, "stderr": stderr})

# --- APKFORGE / FILE OPS ---

@app.route('/api/apkforge/analyze', methods=['POST'])
def apk_analyze():
    apk_path = request.json.get('apk')
    if not os.path.exists(apk_path):
        return jsonify({"error": "File not found"}), 404

    stdout, stderr = run_cmd(f"aapt dump badging {apk_path}")
    return jsonify({"analysis": stdout, "error": stderr})

if __name__ == '__main__':
    # Default to port 8000 to match api-client.js default
    app.run(host='0.0.0.0', port=8000, debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://172.28.0.12:8000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)
